In [2]:
import scipy.io as sio
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 62
N = 517
MAT_FILE = '870d37a7-673d-49ff-8a98-47b9b519237f.mat'
d = sio.loadmat(MAT_FILE)
raw_keys = {
    'Face_C':    'fingToFaceCSCORES', 'Face_G':    'fingToFaceGSCORES',
    'Finger_LI': 'VfingToFaceliSCORES', 'Finger_RI': 'VfingToFaceriSCORES',
}
rng = np.random.default_rng(SEED)
perm = rng.permutation(N)
folds = np.array_split(perm, 5)

def minmax_fit_transform(S_full, train_ids, test_ids):
    S_full = S_full.astype(float)
    S_train = S_full[np.ix_(train_ids, train_ids)]
    S_test  = S_full[np.ix_(test_ids, test_ids)]
    invalid = (S_train == -1)
    valid_vals = S_train[~invalid]
    vmin, vmax = valid_vals.min(), valid_vals.max()
    def transform(S):
        inv = (S == -1)
        out = np.full_like(S, -1.0)
        out[~inv] = (S[~inv] - vmin) / (vmax - vmin)
        return out
    return transform(S_train), transform(S_test)

def rule1(x,y): return x*y*np.maximum(x,y)
def rule2(x,y):
    xy=x*y; return xy*(x+y-xy)
def rule3(x,y): return x*y*np.minimum(x,y)
def rule4(x,y):
    xy=x*y; return xy+xy**2-xy*np.minimum(x,y)
def rule5(x,y):
    s=x+y-x*y; mn=np.minimum(x,y); return s+mn-s*mn
def rule6(x,y):
    xy=x*y; return (x+y)-xy*(x+y-xy)
def rule7(x,y):
    s=x+y-x*y; mx=np.maximum(x,y); return s+mx-s*mx
def rule8(x,y):
    s=x+y-x*y; return s*np.maximum(x,y)

PARAMFREE_RULES = {'Rule1':rule1,'Rule2':rule2,'Rule3':rule3,'Rule4':rule4,
                    'Rule5':rule5,'Rule6':rule6,'Rule7':rule7,'Rule8':rule8}

def apply_rule(rule, x, y):
    valid = (x != -1) & (y != -1)
    xv = np.where(valid, x, 0.0); yv = np.where(valid, y, 0.0)
    fused = PARAMFREE_RULES[rule](xv, yv)
    fused = np.clip(fused, 0.0, 1.0)
    return np.where(valid, fused, -1.0)

SYSTEMS = ['S1','S2','S3','S4','S5','S6','S7','S8']
BASELINE_NAMES = ['Face_C','Face_G','Finger_LI','Finger_RI']

# Winning param-free rule per system, from the seed=62 TRAIN run
PF_WINNERS = {'S1':'Rule3','S2':'Rule5','S3':'Rule5','S4':'Rule5',
              'S5':'Rule5','S6':'Rule8','S7':'Rule5','S8':'Rule5'}

def get_matrix(norm, system, rule):
    if system == 'S1': return apply_rule(rule, norm['Face_C'], norm['Face_G'])
    if system == 'S2': return apply_rule(rule, norm['Face_C'], norm['Finger_LI'])
    if system == 'S3': return apply_rule(rule, norm['Face_C'], norm['Finger_RI'])
    if system == 'S4': return apply_rule(rule, norm['Face_G'], norm['Finger_LI'])
    if system == 'S5': return apply_rule(rule, norm['Face_G'], norm['Finger_RI'])
    if system == 'S6': return apply_rule(rule, norm['Finger_LI'], norm['Finger_RI'])
    if system == 'S7':
        st1 = apply_rule(rule, norm['Finger_LI'], norm['Finger_RI'])
        return apply_rule(rule, st1, norm['Face_C'])
    if system == 'S8':
        st1 = apply_rule(rule, norm['Finger_LI'], norm['Finger_RI'])
        return apply_rule(rule, st1, norm['Face_G'])

def genuine_impostor(S):
    n_ = S.shape[0]
    diag = np.diag(S)
    off_mask = ~np.eye(n_, dtype=bool)
    genuine = diag[diag != -1]
    imp_all = S[off_mask]
    impostor = imp_all[imp_all != -1]
    return genuine, impostor

def ranks_of(S):
    n_ = S.shape[0]
    diag = np.diag(S)
    ranks = []
    for i in range(n_):
        if diag[i] == -1: continue
        row = S[i, :]; valid = row != -1; g = row[i]
        better = np.sum(row[valid] > g)
        ranks.append(better + 1)
    return np.array(ranks)

SYSTEM_ORDER = BASELINE_NAMES + SYSTEMS
pooled = {name: {'genuine':[], 'impostor':[], 'ranks':[]} for name in SYSTEM_ORDER}

for fold_idx in range(5):
    test_ids = np.sort(folds[fold_idx])
    train_ids = np.sort(np.setdiff1d(np.arange(N), test_ids))
    norm_test = {}
    for name, key in raw_keys.items():
        _, Ste = minmax_fit_transform(d[key], train_ids, test_ids)
        norm_test[name] = Ste

    for name in BASELINE_NAMES:
        S = norm_test[name]
        g,i = genuine_impostor(S); r = ranks_of(S)
        pooled[name]['genuine'].append(g); pooled[name]['impostor'].append(i); pooled[name]['ranks'].append(r)

    for system in SYSTEMS:
        rule = PF_WINNERS[system]
        Spf = get_matrix(norm_test, system, rule)
        g,i = genuine_impostor(Spf); r = ranks_of(Spf)
        pooled[system]['genuine'].append(g); pooled[system]['impostor'].append(i); pooled[system]['ranks'].append(r)

for name in pooled:
    pooled[name]['genuine'] = np.concatenate(pooled[name]['genuine'])
    pooled[name]['impostor'] = np.concatenate(pooled[name]['impostor'])
    pooled[name]['ranks'] = np.concatenate(pooled[name]['ranks'])

print('Pooled sizes S1:', len(pooled['S1']['genuine']), 'genuine,', len(pooled['S1']['impostor']), 'impostor')

# ---------------- Plotting ----------------
def roc_curve(genuine, impostor, far_grid):
    imp_sorted = np.sort(impostor)
    gar = []
    for far in far_grid:
        t = np.percentile(imp_sorted, 100*(1-far))
        gar.append(np.mean(genuine >= t))
    return np.array(gar)

def cmc_curve(ranks, rank_grid):
    return np.array([(ranks <= k).mean() for k in rank_grid])

FAR_GRID = np.logspace(-4, 0, 80)
RANK_GRID = np.arange(1, 51)

LEGEND = {n: f'{n} (monomodal)' for n in BASELINE_NAMES}
for s,r in PF_WINNERS.items(): LEGEND[s] = f'{s} \u2013 {r}'

roc_data = {name: roc_curve(pooled[name]['genuine'], pooled[name]['impostor'], FAR_GRID) for name in SYSTEM_ORDER}
cmc_data = {name: cmc_curve(pooled[name]['ranks'], RANK_GRID) for name in SYSTEM_ORDER}

all_roc_y = np.concatenate(list(roc_data.values()))
roc_ymin, roc_ymax = all_roc_y.min(), all_roc_y.max()
roc_pad = 0.02*(roc_ymax-roc_ymin)
roc_ylim = (max(0, roc_ymin-roc_pad), min(1, roc_ymax+roc_pad))

all_cmc_y = np.concatenate(list(cmc_data.values()))
cmc_ymin, cmc_ymax = all_cmc_y.min(), all_cmc_y.max()
cmc_pad = 0.02*(cmc_ymax-cmc_ymin)
cmc_ylim = (max(0, cmc_ymin-cmc_pad), min(1, cmc_ymax+cmc_pad))

print('ROC y range', roc_ymin, roc_ymax, '-> ylim', roc_ylim)
print('CMC y range', cmc_ymin, cmc_ymax, '-> ylim', cmc_ylim)

baseline_colors = {'Face_C':'#7f7f7f','Face_G':'#4d4d4d','Finger_LI':'#bdbdbd','Finger_RI':'#000000'}
fused_cmap = plt.get_cmap('tab10')
fused_colors = {s: fused_cmap(i) for i,s in enumerate(SYSTEMS)}

def style_for(name):
    if name in BASELINE_NAMES:
        return dict(color=baseline_colors[name], linestyle='--', linewidth=2.0, alpha=0.85)
    return dict(color=fused_colors[name], linestyle='-', linewidth=2.8, alpha=0.95)

plt.rcParams.update({'font.size': 14})

fig, ax = plt.subplots(figsize=(13,9.5), dpi=150)
for name in SYSTEM_ORDER:
    ax.plot(FAR_GRID, roc_data[name], label=LEGEND[name], **style_for(name))
ax.set_xscale('log')
ax.set_xlim(FAR_GRID.min(), FAR_GRID.max())
ax.set_ylim(*roc_ylim)
ax.set_xlabel('False Accept Rate (FAR)', fontsize=19, fontweight='bold', labelpad=10)
ax.set_ylabel('Genuine Accept Rate (GAR = 1 - FRR)', fontsize=19, fontweight='bold', labelpad=10)
ax.set_title('ROC Curves \u2014 Parameter-Free Rules at Optimal Rule (pooled 5-fold TEST, seed=62)', fontsize=19, fontweight='bold', pad=16)
ax.tick_params(axis='both', labelsize=15)
ax.grid(True, which='both', linestyle=':', alpha=0.5)
ax.axvline(0.01, color='red', linestyle='--', linewidth=2.2, alpha=0.85)
ax.text(0.0105, roc_ylim[0]+0.015*(roc_ylim[1]-roc_ylim[0]), 'FAR = 1%', color='red', fontsize=14,
        rotation=90, va='bottom', fontweight='bold')
leg = ax.legend(loc='lower right', fontsize=14, ncol=2, framealpha=0.95,
                 title='System \u2013 Rule', title_fontsize=16, borderpad=1.0, labelspacing=0.65)
leg.get_title().set_fontweight('bold')
fig.tight_layout()
fig.savefig('./Results/ROC_ParamFree_TEST_seed62.png')
#plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(13,9.5), dpi=150)
for name in SYSTEM_ORDER:
    ax.plot(RANK_GRID, cmc_data[name], label=LEGEND[name], **style_for(name))
ax.set_xlim(1, RANK_GRID.max())
ax.set_ylim(*cmc_ylim)
ax.set_xlabel('Rank', fontsize=19, fontweight='bold', labelpad=10)
ax.set_ylabel('Cumulative Identification Rate', fontsize=19, fontweight='bold', labelpad=10)
ax.set_title('CMC Curves \u2014 Parameter-Free Rules at Optimal Rule (pooled 5-fold TEST, seed=62)', fontsize=19, fontweight='bold', pad=16)
ax.tick_params(axis='both', labelsize=15)
ax.grid(True, linestyle=':', alpha=0.5)
leg = ax.legend(loc='lower right', fontsize=14, ncol=2, framealpha=0.95,
                 title='System \u2013 Rule', title_fontsize=16, borderpad=1.0, labelspacing=0.65)
leg.get_title().set_fontweight('bold')
fig.tight_layout()
fig.savefig('./Results/CMC_ParamFree_TEST_seed62.png')
#plt.show()
plt.close(fig)
print('done')

Pooled sizes S1: 516 genuine, 52840 impostor
ROC y range 0.6769825918762089 1.0 -> ylim (np.float64(0.670522243713733), 1)
CMC y range 0.8839458413926499 1.0 -> ylim (np.float64(0.8816247582205029), 1)


C:\Users\Fdl\AppData\Local\Temp\ipykernel_33576\2843603480.py:192: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


done


C:\Users\Fdl\AppData\Local\Temp\ipykernel_33576\2843603480.py:210: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
